# 11 – Performance baseline

Amaç: Tuning öncesi 3 sorgunun (join, aggregation, tarih filtresi) planını ve süresini `trans_big` ve `trans_skewed` üzerinde ölçmek.

Ölçüm araçları: `EXPLAIN`, Query Profile ("See performance") ve süre. Free Edition'da Spark UI yok.

Bu notebook hiçbir tablo yazmaz; sadece okur.

## 1. Başlangıç durumu

Üç tablonun dosya sayısı, boyutu, clustering kolonları ve satır sayısı.

Nasıl okunur: `numFiles` 1 ve `size_mb` ~26–30 ise veri tek dosyada demektir; okuma büyük ihtimalle tek task ile yapılır. `clusteringColumns` serverless'ın otomatik seçtiği kolonları gösterir (boşsa yok).

In [0]:
from pyspark.sql import functions as F

PERF_SCHEMA = "workspace.perf"
TRANS_TABLES = ["trans_big", "trans_skewed"]
ACCOUNT_TABLE = "workspace.silver.account"

all_tables = []
for table_name in TRANS_TABLES:
    all_tables.append(f"{PERF_SCHEMA}.{table_name}")
all_tables.append(ACCOUNT_TABLE)

df_details = None
for full_name in all_tables:
    # DESCRIBE DETAIL: baseline'ın fiziksel başlangıç durumu (dosya, boyut, clustering)
    df_detail = spark.sql(f"DESCRIBE DETAIL {full_name}")
    df_detail = df_detail.select(
        F.lit(full_name).alias("table_name"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
        "partitionColumns",
        "clusteringColumns",
    )
    row_count = spark.table(full_name).count()
    df_detail = df_detail.withColumn("rows", F.lit(row_count))

    if df_details is None:
        df_details = df_detail
    else:
        df_details = df_details.unionByName(df_detail)

display(df_details)

## 2. Sorgular ve fiziksel plan (EXPLAIN)

Üç sorgu function olarak tanımlanır; parametre trans tablosunun adıdır. Böylece aynı sorgu `trans_big` ve `trans_skewed` üzerinde çalışır.

`explain()` sorguyu çalıştırmaz, sadece Spark'ın planını gösterir. Plan iki tabloda aynı çıkacağı için sadece `trans_big` ile bakılır; skew farkı süre ve Query Profile'da görülür.

Nasıl okunur:

- Q1: `BroadcastHashJoin` (küçük tablo her task'a kopyalanır, shuffle yok) mı, `SortMergeJoin` (iki taraf da shuffle edilir) mı?

- Q2: `Exchange` / `Shuffle` satırı: `account_id`'ye göre veri yeniden dağıtılıyor.

- Q3: Scan satırındaki `PushedFilters` / `DataFilters`: filtre okuma seviyesine itilmiş mi?

- `AdaptiveSparkPlan isFinalPlan=false`: AQE planı çalışma sırasında değiştirebilir; kesin plan Query Profile'da.

- Node adları `Photon...` ile başlıyorsa sorgu Photon motorunda çalışıyor (serverless varsayılanı).

In [0]:
def q1_join(trans_table):
    df_trans = spark.table(trans_table)
    # account küçük tablo (4.500 satır): broadcast join adayı
    df_account = spark.table(ACCOUNT_TABLE).select("account_id", "district_id")
    df_joined = df_trans.join(df_account, "account_id")
    df_result = df_joined.groupBy("district_id").agg(
        F.count("*").alias("trans_count"),
        F.sum("amount").alias("total_amount"),
    )
    return df_result


def q2_agg(trans_table):
    df_trans = spark.table(trans_table)
    # account_id'ye göre gruplama: skew'lü tabloda hot account 8261 tek task'a düşer
    df_result = df_trans.groupBy("account_id").agg(
        F.count("*").alias("trans_count"),
        F.sum("amount").alias("total_amount"),
    )
    return df_result


def q3_filter(trans_table):
    df_trans = spark.table(trans_table)
    # tek yıl: data skipping için ölçü (dosya tek olduğu için şu an atlanacak dosya yok)
    df_filtered = df_trans.filter(
        (F.col("trans_date") >= "2020-01-01") & (F.col("trans_date") < "2021-01-01")
    )
    df_result = df_filtered.agg(
        F.count("*").alias("trans_count"),
        F.sum("amount").alias("total_amount"),
    )
    return df_result


QUERIES = {
    "q1_join": q1_join,
    "q2_agg": q2_agg,
    "q3_filter": q3_filter,
}

big_table = f"{PERF_SCHEMA}.trans_big"
for query_name in QUERIES:
    query_function = QUERIES[query_name]
    print("=" * 80)
    print(query_name)
    print("=" * 80)
    query_function(big_table).explain()

## 3. Süre ölçümü

Her sorgu, her iki tabloda 3 kez çalıştırılır. Süre `collect()` ile ölçülür; sonuçlar küçük olduğu için (en fazla ~4.500 satır) driver'a çekmek sorun değil.

Nasıl okunur:

- `run_no` 1 genelde daha yavaştır (ısınma, cache). Karşılaştırmada `min_s` kullanılır.

- Süreye Spark Connect'in ağ gidiş-dönüşü de dahil; küçük farklar (< 0,5 sn) anlamlı sayılmaz.

- `result_rows`: Q1 district sayısı (77), Q2 hesap sayısı (~4.500), Q3 tek satır.

In [0]:
import time

RUNS = 3

timing_rows = []
for table_name in TRANS_TABLES:
    full_name = f"{PERF_SCHEMA}.{table_name}"
    for query_name in QUERIES:
        query_function = QUERIES[query_name]
        for run_no in range(1, RUNS + 1):
            df_query = query_function(full_name)
            # sadece collect süresi ölçülür; plan kurma ölçüme girmez
            start = time.perf_counter()
            result_rows = df_query.collect()
            seconds = time.perf_counter() - start
            timing_rows.append((table_name, query_name, run_no, len(result_rows), round(seconds, 2)))

df_timings = spark.createDataFrame(
    timing_rows,
    ["table_name", "query_name", "run_no", "result_rows", "seconds"],
)

df_summary = (
    df_timings.groupBy("query_name", "table_name")
    .agg(
        F.min("seconds").alias("min_s"),
        F.round(F.avg("seconds"), 2).alias("avg_s"),
        F.max("seconds").alias("max_s"),
    )
    .orderBy("query_name", "table_name")
)

display(df_timings)
display(df_summary)